# Cervical Cancer Detection - Clinical Training (T4 GPU Optimized)

This notebook trains a production-grade cervical cancer classification model optimized for Google Colab **T4 GPU** (free tier).

**Clinical Targets** (from published AI cervical cytology benchmarks):
- ✅ **Sensitivity**: ≥95% (CRITICAL - minimize missed cases)
- ✅ **Specificity**: ≥94%
- ✅ **Accuracy**: ≥94%
- ✅ **PPV**: ≥88%
- ✅ **NPV**: ≥95%

**Training Time**:
- T4 GPU: ~4-6 hours (Phase 1: 15 epochs + Phase 2: 35 epochs)
- Auto-checkpointing every 5 epochs for session recovery

---

## 1. Setup Environment

### 1.1 Check GPU Availability

In [ ]:
import tensorflow as tf

# Check GPU
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"GPU Found: {gpus}")
    print(f"  GPU Details: {tf.config.experimental.get_device_details(gpus[0])}")
    print("\n   T4 GPU Optimizations Enabled:")
    print("     • Mixed Precision FP16: 1.5-2x speedup")
    print("     • XLA Compilation: +15% performance")
    print("     • Batch size: 40 (optimal for 16GB VRAM)")
else:
    print(" NO GPU FOUND! Training will be VERY slow on CPU.")
    print("   Please enable GPU: Runtime > Change runtime type > GPU")

### 1.2 Clone Repository from GitHub

**Why clone?** This notebook runs in **Google Colab** (cloud), not on your local machine. 
We need to get your code from GitHub into the Colab environment.

**Note**: If you're running this locally (not in Colab), skip this cell!

In [ ]:
import os

# Check if we're in Colab or local environment
try:
    from google.colab import drive
    IN_COLAB = True
    print("✅ Running in Google Colab")
except ImportError:
    IN_COLAB = False
    print("✅ Running locally - skipping clone")

if IN_COLAB:
    repo_name = "Automated-Cervical-Cancer-Detection-via-Deep-Learning"
    
    if not os.path.exists(repo_name):
        print("📥 Cloning repository from GitHub to Colab...")
        !git clone https://github.com/NgangaKamau3/Automated-Cervical-Cancer-Detection-via-Deep-Learning.git
        print("✅ Repository cloned")
    else:
        print("✅ Repository already exists")
    
    # Change to project directory
    %cd Automated-Cervical-Cancer-Detection-via-Deep-Learning
else:
    print("💡 Make sure you're in the project directory!")
    print(f"   Current directory: {os.getcwd()}")

### 1.3 Install Dependencies

In [ ]:
print("Installing dependencies...")
%pip install -q -r requirements.txt
print("Dependencies installed")

### 1.4 Mount Google Drive (for checkpoints & dataset storage)

In [ ]:
from google.colab import drive

drive.mount('/content/drive', force_remount=False)
print("✅ Google Drive mounted")

# Create project directory in Drive for persistent storage
drive_project_dir = '/content/drive/MyDrive/cervical_cancer_models'
os.makedirs(drive_project_dir, exist_ok=True)
print(f"✅ Drive storage ready: {drive_project_dir}")

---

## 2. Download Dataset

We need to authenticate with Kaggle to download the dataset.

**Option A (Recommended): Colab Secrets**
1. In the sidebar on the left, click the Key icon (Secrets).
2. Add two secrets: `KAGGLE_USERNAME` and `KAGGLE_KEY`.
3. Toggle "Notebook access" ON for both.

**Option B: Upload kaggle.json**
If you don't use secrets, you'll be prompted to upload `kaggle.json`.

In [ ]:
import os
from google.colab import files
from google.colab import userdata

# Try to get credentials from Colab Secrets
try:
    os.environ["KAGGLE_USERNAME"] = userdata.get('KAGGLE_USERNAME')
    os.environ["KAGGLE_KEY"] = userdata.get('KAGGLE_KEY')
    print("✅ Kaggle credentials loaded from Colab Secrets")
    has_auth = True
except Exception:
    print("⚠️ Colab Secrets not found or not accessible.")
    has_auth = False

# Fallback to kaggle.json upload
if not has_auth:
    kaggle_dir = os.path.expanduser('~/.kaggle')
    kaggle_json = os.path.join(kaggle_dir, 'kaggle.json')

    if not os.path.exists(kaggle_json):
        print("\n📤 Please upload your kaggle.json file:")
        uploaded = files.upload()
        
        if 'kaggle.json' in uploaded:
            os.makedirs(kaggle_dir, exist_ok=True)
            !mv kaggle.json ~/.kaggle/
            !chmod 600 ~/.kaggle/kaggle.json
            print("✅ Kaggle configured via file upload")
    else:
        print("✅ Kaggle already configured (kaggle.json found)")

In [ ]:
# Run the download script
!python scripts/download_dataset.py

---

## 3. Train Model

Runs the production training script with:
- **T4 GPU optimizations** (mixed precision, XLA, batch size 40)
- **Focal loss** for sensitivity maximization
- **Clinical metrics** (sensitivity, specificity, PPV, NPV, AUC)
- **Aggressive checkpointing** (every 5 epochs)
- **Two-phase training**: Frozen base (15 epochs) + Fine-tuning (35 epochs)

### 3.1 Configure Training (Optional: Modify Parameters)

In [ ]:
# Training configuration
ARCHITECTURE = 'efficientnet'  # Options: 'efficientnet' or 'resnet'
BATCH_SIZE = 40                # Optimized for T4 GPU (16GB VRAM)
INITIAL_EPOCHS = 15            # Phase 1: frozen base
FINE_TUNE_EPOCHS = 35          # Phase 2: fine-tuning (total ~50 epochs)

# Symlink models directory to Google Drive for persistence
models_dir = './models'
drive_models_dir = f'{drive_project_dir}/models'

if os.path.exists(models_dir) and not os.path.islink(models_dir):
    !rm -rf {models_dir}

if not os.path.exists(models_dir):
    os.makedirs(drive_models_dir, exist_ok=True)
    !ln -s {drive_models_dir} {models_dir}
    print(f"✅ Models will be saved to Google Drive: {drive_models_dir}")
else:
    print(" Using existing models directory in Drive")

### 3.2 Start Training 🚀

**Expected Time**: 4-6 hours on T4 GPU

**What to expect**:
1. Phase 1 (15 epochs, ~1-2 hours): Train with frozen EfficientNetB3 base
2. Phase 2 (35 epochs, ~3-4 hours): Fine-tune top layers
3. Clinical evaluation with benchmark comparison
4. Auto-save checkpoints every 5 epochs to Google Drive

In [ ]:
# Run training script
!python ml/train.py \
    --architecture {ARCHITECTURE} \
    --batch-size {BATCH_SIZE} \
    --initial-epochs {INITIAL_EPOCHS} \
    --fine-tune-epochs {FINE_TUNE_EPOCHS}

---

## 4. Evaluate Results

### 4.1 View Clinical Report

In [ ]:
import json

# Load clinical report
with open('models/clinical_report.json', 'r') as f:
    clinical_report = json.load(f)

# Display key metrics
overall = clinical_report['clinical_performance']['overall_metrics']

print("=" * 60)
print("  CLINICAL PERFORMANCE SUMMARY")
print("=" * 60)
print(f"Sensitivity: {overall['macro_sensitivity']:.1%} (95% CI: {overall['macro_sensitivity_ci'][0]:.1%}-{overall['macro_sensitivity_ci'][1]:.1%})")
print(f"Specificity: {overall['macro_specificity']:.1%} (95% CI: {overall['macro_specificity_ci'][0]:.1%}-{overall['macro_specificity_ci'][1]:.1%})")
print(f"Accuracy:    {overall['accuracy']:.1%} (95% CI: {overall['accuracy_ci'][0]:.1%}-{overall['accuracy_ci'][1]:.1%})")
print(f"PPV:         {overall['macro_ppv']:.1%}")
print(f"NPV:         {overall['macro_npv']:.1%}")
print(f"\nWeakest Class: {overall['weakest_class']} (Sensitivity: {overall['min_sensitivity']:.1%})")

print("\n" + "=" * 60)
print("  CLINICAL INTERPRETATION")
print("=" * 60)
for interpretation in clinical_report['clinical_interpretation']:
    print(f"  {interpretation}")

### 4.2 Display Visualizations

In [ ]:
from IPython.display import Image, display

print("📊 ROC Curves:")
display(Image('models/roc_curves.png'))

print("\n📊 Confusion Matrix with Clinical Metrics:")
display(Image('models/confusion_matrix_clinical.png'))

print("\n📈 Training History (Phase 1):")
display(Image('models/training_history_phase1.png'))

print("\n📈 Training History (Phase 2):")
display(Image('models/training_history_phase2.png'))

---

## 5. Export Model for Deployment

In [ ]:
# Export final model to SavedModel format
!python ml/export.py \
    --model-path models/efficientnet_final.keras \
    --export-all

print("✅ Model exported to models/export/")
print("\nExported formats:")
print("  • SavedModel (for production API)")
print("  • TensorFlow Lite (for mobile/edge)")
print("  • Model metadata (classes, config)")

---

## 6. Download Models and Results

### 6.1 Compress Models

In [ ]:
import shutil

# Create archive
print("📦 Compressing models and results...")
shutil.make_archive('cervical_cancer_models', 'zip', 'models')
print("✅ Archive created: cervical_cancer_models.zip")

# Show file size
size_mb = os.path.getsize('cervical_cancer_models.zip') / (1024 * 1024)
print(f"   Size: {size_mb:.1f} MB")

### 6.2 Download to Local Machine

In [ ]:
from google.colab import files

# Download archive
print("📥 Starting download...")
files.download('cervical_cancer_models.zip')
print("✅ Download complete!")
print("\nYour models are also saved in Google Drive at:")
print(f"  {drive_models_dir}")

---

## 7. Resume Training (If Session Disconnected)

If your Colab session disconnects before training completes, you can resume from the last checkpoint:

In [ ]:
# List available checkpoints
import glob

checkpoints = sorted(glob.glob('models/checkpoint_*.keras'))
if checkpoints:
    print("📁 Available checkpoints:")
    for cp in checkpoints:
        size_mb = os.path.getsize(cp) / (1024 * 1024)
        print(f"  • {os.path.basename(cp)} ({size_mb:.1f} MB)")
    
    latest = checkpoints[-1]
    print("\n To resume training, modify the train.py script to load:")
    print(f"   {latest}")
else:
    print("⚠️ No checkpoints found")

---

## ✅ Training Complete!

**Next Steps**:
1. ✅ Review clinical metrics (should meet ≥95% sensitivity)
2. ✅ Download models from Google Drive
3. ✅ Deploy using Docker:
   ```bash
   docker-compose up -d
   ```
4. ✅ Test API:
   ```bash
   curl -X POST "http://localhost:8000/predict" -F "file=@test_image.jpg  "
   ```

**Model Files**:
- `efficientnet_final.keras` - Best model from training
- `export/saved_model/` - Production deployment format
- `export/model.tflite` - Mobile/edge deployment
- `clinical_report.json` - Comprehensive clinical validation

---

**⚕️ Medical Disclaimer**: This software is for research and educational purposes only. It is not intended for clinical diagnostic use. Always consult with qualified medical professionals for medical advice and diagnosis.